In [36]:
import numpy as np
import matplotlib.pyplot as plt
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from ngsolve.solvers import NewtonMinimization
import ipywidgets as widgets
from ipywidgets import VBox, HTML

In [37]:
def C(u):
    F = Id(2) + Grad(u)
    return F.trans * F

def neo_hookean (C, lam, mu):
    return 0.5*mu*(Trace(C-Id(2)) + 2*mu/lam*Det(C)**(-lam/2/mu)-1)

# def stress_neo_hookean(C, lam, mu):
#     return 0.5 * mu * (Id(2) - Det(C) ** (-lam / 2 / mu) * Inv(C))

In [38]:
# Create Unit Square Mesh
mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
Draw(mesh, mesh=True)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [39]:


fes = VectorH1(mesh, order=2, dirichlet="bottom", dim=mesh.dim)

u = fes.TrialFunction()
v = fes.TestFunction()

E = 2100000
nu = 0.2
mu = E / (2 * (1 + nu))
lam = E * nu / ((1 + nu) * (1 - 2 * nu))

n = specialcf.normal(mesh.dim)
traction_magnitude = Parameter(1)
traction_top = traction_magnitude * n

a = BilinearForm(fes)

a += Variation(neo_hookean(C(u), lam, mu).Compile() * dx)
a += Variation(-InnerProduct(traction_top, u).Compile() * ds("top"))

gfu = GridFunction(fes)
gfu.vec[:] = 0

scene = Draw(gfu, mesh)

with TaskManager():
    for s in np.linspace(0, 1, 10):     
        traction_magnitude.Set(-1.0 * s)            
        NewtonMinimization(a=a, u=gfu, printing=False, inverse="sparsecholesky")
        scene.Redraw()

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

In [ ]:
# --- mesh & space (2D) ---
mesh = Mesh(unit_square.GenerateMesh(maxh=0.08))
V = VectorH1(mesh, order=2, dirichlet="bottom|left")   # clamp some boundary to kill rigid body modes
u, v = V.TnT()
gfu = GridFunction(V)

# --- material parameters (plane formulation) ---
E, nu = 210.0, 0.3
lam = (E*nu)/((1+nu)*(1-2*nu))
mu  = E/(2*(1+nu))

# --- kinematics & energy ---
def F(u):  return Id(mesh.dim) + Grad(u)
def psi(u):
    C = F(u).trans * F(u)
    J = Det(F(u))
    I1 = Trace(C)
    # Ψ = μ/2(I1−3) − μ ln J + λ/2 (ln J)^2  (total Lagrangian)
    return 0.5*mu*(I1-3) - mu*log(J) + 0.5*lam*(log(J))**2

# --- traction on boundary named "force" ---
# mark the top boundary as "force" when you build the mesh (or rename appropriately here)
n = specialcf.normal(mesh.dim)      # reference outward normal
p0 = 1e6                         # traction magnitude (force per reference length)
loadpar = Parameter(1.0)
tref = loadpar * p0 * n             # pure normal traction

# --- total potential Π(u) = ∫Ω Ψ dΩ − ∫Γ t·u ds ---
a = BilinearForm(V, symmetric=True)
a += Variation(psi(u).Compile() * dx)
a += Variation(-InnerProduct(tref, u).Compile() * ds("force"))

# --- solve with load stepping ---
gfu.vec[:] = 0.0
scene = Draw(gfu, mesh, "Deformed configuration")
with TaskManager():
    for s in np.linspace(0, 1, 11):
        loadpar.Set(s)
        NewtonMinimization(a=a, u=gfu, printing=False, inverse="sparsecholesky")
        # Show deformed configuration
        scene.Redraw()

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…